## residual block с двумя cnn-слоями

In [ ]:
import torch.nn as nn
from torch import Tensor


class ResidualBlock(nn.Module):

    def __init__(
        self,
        inputs: int,
        outputs: int,
        stride: int = 1,
    ) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(inputs,
                               outputs,
                               kernel_size=3,
                               stride=stride,
                               bias=False,
                               padding='same')
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(outputs,
                               outputs,
                               kernel_size=3,
                               stride=stride,
                               bias=False,
                               padding='same')

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.relu(out)

        out = self.conv2(out)

        out += identity
        out = self.relu(out)

        return out

## residial block с трямя cnn-слоями и другим числом нейронов (bottleneck) на среднем из них

In [ ]:
import torch.nn as nn
from torch import Tensor


class ResidualBlock(nn.Module):

    def __init__(
        self,
        inputs: int,
        hidden: int,
        outputs: int,
        stride: int = 1,
    ) -> None:
        super().__init__()

        self.conv1 = nn.Conv2d(inputs,
                               hidden,
                               kernel_size=3,
                               stride=stride,
                               bias=False,
                               padding='same')
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(hidden,
                               hidden,
                               kernel_size=3,
                               stride=stride,
                               bias=False,
                               padding='same')
        self.conv3 = nn.Conv2d(hidden,
                               outputs,
                               kernel_size=3,
                               stride=stride,
                               bias=False,
                               padding='same')

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.relu(out)

        out = self.conv3(out)

        out += identity               # здесь добавляется residual connection
        out = self.relu(out)

        return out


print(ResidualBlock(256, 64, 256))

ResidualBlock(
  (conv1): Conv2d(256, 64, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
  (relu): ReLU()
  (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
  (conv3): Conv2d(64, 256, kernel_size=(3, 3), stride=(1, 1), padding=same, bias=False)
)


## residual block с batch normalization

In [ ]:
import torch.nn as nn
from torch import Tensor


class ResidualBlock(nn.Module):

    def __init__(
        self,
        inputs: int,
        outputs: int,
        stride: int = 1,
    ) -> None:
        super().__init__()
        norm_layer = nn.BatchNorm2d
        self.conv1 = nn.Conv2d(inputs,
                               outputs,
                               kernel_size=3,
                               stride=stride,
                               bias=False,
                               padding='same')
        self.bn1 = norm_layer(outputs)
        # Можно объявить один раз и переиспользовать,
        # так как в reLU нет обучаемых параметров
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(outputs,
                               outputs,
                               kernel_size=3,
                               stride=stride,
                               bias=False,
                               padding='same')
        self.bn2 = norm_layer(outputs)

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)

        out += identity
        out = self.relu(out)

        return out

Говоря о слое батч-нормализации, нужно учитывать его взаимодействие с другой техникой регуляризации — Dropout. Интересно, что в ResNet не используется Dropout, хотя ещё в VGG он был важным компонентом полносвязной части модели. Это происходит по двум причинам.
Батч-нормализация сама по себе помогает предотвращать обучение. Поэтому Dropout ей просто может быть не нужен.
Иногда Dropout все-таки нужен, и тогда исследователи рекомендуют размещать его после всех слоёв батч-нормализации. Если разместить Dropout перед батч-нормализацией, то во время обучения он будет отключать случайную часть активаций. Из-за этого он будет менять параметры распределения их значений, которые сохраняются как скользящие среднее и стандартное отклонение в слое BatchNorm. А при использовании обученной модели Dropout работать уже не будет, соответственно, параметры распределения активаций будут различаться, а BatchNorm сохранял и ориентировался на прежние — с Dropout. Из-за этого может упасть точность распознавания при тестировании модели. **Поэтому Dropout при необходимости располагается только после слоя батч-нормализации, а для надёжности — после всех слоёв батч-нормализации**.

## реализация bottle neck точечной сверткой (kernel = 1x1) 

In [ ]:
import torch.nn as nn
from torch import Tensor


class BottleneckBlock(nn.Module):

    def __init__(
        self,
        inputs: int,
        hidden: int,
        outputs: int,
        stride: int = 1,
    ) -> None:
        super().__init__()
        norm_layer = nn.BatchNorm2d
        self.conv1 = nn.Conv2d(inputs,
                               hidden,
                               kernel_size=1,
                               stride=stride,
                               bias=False,
                               padding='same')
        self.bn1 = norm_layer(hidden)
        self.relu = nn.ReLU()
        self.conv2 = nn.Conv2d(hidden,
                               hidden,
                               kernel_size=3,
                               stride=stride,
                               bias=False,
                               padding='same')
        self.bn2 = norm_layer(outputs)
        self.conv3 = nn.Conv2d(hidden,
                               outputs,
                               kernel_size=1,
                               stride=stride,
                               bias=False,
                               padding='same')
        self.bn3 = norm_layer(outputs)

    def forward(self, x: Tensor) -> Tensor:
        identity = x

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        out += identity
        out = self.relu(out)

        return out


model = BottleneckBlock(256, 64, 256)
print("Архитектура блока:", model)
print("Количество параметров:", sum(p.numel() for p in model.parameters()))